In [1]:
import pyspark

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
        .appName('test') \
            .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/08 18:23:51 WARN Utils: Your hostname, codespaces-00ce04, resolves to a loopback address: 127.0.0.1; using 10.0.11.97 instead (on interface eth0)
26/11/08 18:23:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/08 18:23:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df_green=spark.read.parquet('data/pq/green/*/*')
df_yellow=spark.read.parquet('data/pq/yellow/*/*')

26/11/08 18:23:59 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/pq/green/*/*.
java.io.FileNotFoundException: File data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.analysis.ResolveData

In [3]:
# with this code we can see cols that intersect with each other 
set(df_green.columns) & set(df_yellow.columns)

{'DOLocationID',
 'PULocationID',
 'RatecodeID',
 'VendorID',
 'congestion_surcharge',
 'extra',
 'fare_amount',
 'improvement_surcharge',
 'mta_tax',
 'passenger_count',
 'payment_type',
 'store_and_fwd_flag',
 'tip_amount',
 'tolls_amount',
 'total_amount',
 'trip_distance'}

In [4]:
df_green=df_green \
    .withColumnRenamed('lpep_pickup_datetime','pickup_datetime') \
        .withColumnRenamed('lpep_dropoff_datetime','dropoff_datetime')


df_yellow=df_yellow \
    .withColumnRenamed('tpep_pickup_datetime','pickup_datetime') \
        .withColumnRenamed('tpep_dropoff_datetime','dropoff_datetime')
        

In [5]:
set(df_green.columns) & set(df_yellow.columns)

{'DOLocationID',
 'PULocationID',
 'RatecodeID',
 'VendorID',
 'congestion_surcharge',
 'dropoff_datetime',
 'extra',
 'fare_amount',
 'improvement_surcharge',
 'mta_tax',
 'passenger_count',
 'payment_type',
 'pickup_datetime',
 'store_and_fwd_flag',
 'tip_amount',
 'tolls_amount',
 'total_amount',
 'trip_distance'}

In [6]:
common_cols = []

for col in set(df_yellow.columns):
    if col in set(df_green.columns):
        common_cols.append(col)


In [7]:
common_cols

['VendorID',
 'extra',
 'congestion_surcharge',
 'payment_type',
 'PULocationID',
 'tip_amount',
 'DOLocationID',
 'store_and_fwd_flag',
 'tolls_amount',
 'passenger_count',
 'mta_tax',
 'fare_amount',
 'total_amount',
 'improvement_surcharge',
 'RatecodeID',
 'trip_distance',
 'dropoff_datetime',
 'pickup_datetime']

In [8]:
from pyspark.sql import functions as F
# F.lit just creates the same name of constant thinsg into that column it means literal
df_green_sel=df_green.select(common_cols) \
    .withColumn('service_type',F.lit('green'))

df_yellow_sel=df_yellow.select(common_cols) \
    .withColumn('service_type',F.lit('yellow'))


In [9]:
df_trips_data = df_green_sel.unionAll(df_yellow_sel)


In [10]:
df_trips_data.groupBy('service_type').count().show()

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2802931|
|      yellow|55553400|
+------------+--------+



In [11]:
df_trips_data.registerTempTable('trips_data')

/workspaces/spark-learn/venv/lib/python3.14/site-packages/pyspark/sql/classic/dataframe.py:186: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [13]:
spark.sql("""
SELECT count(*)
FROM trips_data
GROUP BY service_type
          """).show()

+--------+
|count(1)|
+--------+
| 2802931|
|55553400|
+--------+

